# Notebook 02 — Data PreprocessingTransforms the raw GitHub issues export into the working dataset used by alldownstream notebooks.**Pipeline stages**1. Bot reply filtering2. Label normalisation3. Combined text construction (token-budgeted)4. Column reduction5. Post-construction validation**Inputs**  `data/github_issues_tickets.csv` — 15,955 x 132**Outputs**- `data/github_issues_cleaned.csv` — 15,955 x 133 (all columns + `combined_text`)- `data/github_issues_final.csv` — 15,955 x 37 (working dataset)Section 6 holds data-quality audits reported in the methodology. Those cellsare read-only diagnostics: they never modify the pipeline dataframes.**Run this notebook top to bottom.** No cell reassigns a pipeline variable.

In [1]:
import os
import re
import numpy as np
import pandas as pd

os.chdir('/Users/yaminiumarani/Desktop/rag_thesis_project')
print("Working directory:", os.getcwd())

Working directory: /Users/yaminiumarani/Desktop/rag_thesis_project


In [2]:
df = pd.read_csv('data/github_issues_tickets.csv', low_memory=False)
print("Loaded:", df.shape)

Loaded: (15955, 132)


## 1. Bot reply filteringAutomated accounts post templated triage messages that carry no resolutioncontent. Their replies are treated as noise and removed before textconstruction, so they neither enter the embedded representation nor accrueauthor authority.Detection is a substring match on the reply author's login against fourpatterns. Section 6.3 quantifies the false positives this produces.

In [3]:
bot_patterns = ['bot', 'github-actions', 'dependabot', 'codecov']

def is_bot(author):
    if pd.isna(author):
        return False
    return any(p in str(author).lower() for p in bot_patterns)

# Count matched reply cells per thread position
total_bots = 0
for i in range(10):
    col = f'answers_{i}_author'
    if col in df.columns:
        n = df[col].apply(is_bot).sum()
        total_bots += n
        print(f"Answer {i} - bot replies: {n}")

print(f"\nTotal bot replies to filter: {total_bots}")

Answer 0 - bot replies: 682
Answer 1 - bot replies: 490
Answer 2 - bot replies: 385
Answer 3 - bot replies: 311
Answer 4 - bot replies: 236
Answer 5 - bot replies: 143
Answer 6 - bot replies: 116
Answer 7 - bot replies: 81
Answer 8 - bot replies: 47
Answer 9 - bot replies: 29

Total bot replies to filter: 2520


In [4]:
# Distinct accounts matched by the filter
bot_matched_accounts = set()
for i in range(10):
    col = f'answers_{i}_author'
    if col in df.columns:
        s = df[col].dropna().astype(str)
        bot_matched_accounts.update(s[s.apply(is_bot)].unique())

print("Distinct matched accounts:", len(bot_matched_accounts))
print("Matched without the substring 'bot':",
      sorted(n for n in bot_matched_accounts if 'bot' not in n.lower()))

Distinct matched accounts: 97
Matched without the substring 'bot': []


In [5]:
# Null out author, body and timestamp for every matched reply
for i in range(10):
    author_col = f'answers_{i}_author'
    body_col   = f'answers_{i}_body'
    time_col   = f'answers_{i}_creation_time'

    if author_col in df.columns:
        bot_mask = df[author_col].apply(is_bot)
        df.loc[bot_mask, author_col] = None
        df.loc[bot_mask, body_col]   = None
        df.loc[bot_mask, time_col]   = None

remaining = sum(df[f'answers_{i}_author'].apply(is_bot).sum()
                for i in range(10) if f'answers_{i}_author' in df.columns)
print("Bot replies remaining:", remaining)
assert remaining == 0, "Bot filtering incomplete"
print("Bot filtering complete")

Bot replies remaining: 0
Bot filtering complete


## 2. Label normalisationThe raw export treats `bug` and `Bug` as distinct labels, fragmenting thelabel distribution. Casing and surrounding whitespace are normalised acrossall label columns so that label-based weighting and partitioning operate ona single canonical form.

In [6]:
label_cols = [f'labels_{i}_name' for i in range(11) if f'labels_{i}_name' in df.columns]

for col in label_cols:
    df[col] = df[col].str.lower().str.strip()

print("Label columns normalised:", len(label_cols))
print("\n=== TOP LABELS AFTER NORMALISATION ===")
print(df['labels_0_name'].value_counts().head(10))

Label columns normalised: 11

=== TOP LABELS AFTER NORMALISATION ===
labels_0_name
bug                              12865
enhancement                        540
documentation                      357
good first issue                   211
help wanted                        155
waiting for customer response      133
type: bug                           81
type/bug                            74
issue: ready for work               57
no code attached yet                57
Name: count, dtype: int64


## 3. Combined text constructionEach issue is serialised into a single `combined_text` field holding itstitle, primary label, body and conversation thread.### Token budgeting`all-MiniLM-L6-v2` accepts a maximum of **256 tokens** and silently discardseverything beyond that. Unconstrained concatenation places the bodyimmediately after the title, and issue bodies are long enough that the windowis exhausted before any reply is reached — so no resolution content survivesinto the embedded representation (verified in Section 5.3).Construction is therefore budgeted: each component receives a tokenallocation, and replies consume whatever the earlier components leave. Thefield sits inside the encoder's window by design rather than by chance.| Component | Allocation | Typical use ||---|---|---|| Title | 60 | ~15 || Label | 15 | ~5 || Body | 120 | 120 (most bodies exceed the cap) || Replies | remainder | ~105 |`BODY_BUDGET` governs the trade-off between problem description andresolution content. Lowering it admits more of the thread at the cost ofdetail in the problem statement.### Truncation by character offsetTruncation is applied by **slicing the original string**, not by decodingtruncated token ids back to text. The encoder's tokenizer is uncased andsplits on punctuation, so a decode round-trip is lossy: it lowercases thetext and inserts spaces around punctuation, which would corrupt technicalidentifiers (`Ember.onerror`, `IsResizable`) and the structural field markersthe representation depends on.Instead each segment is tokenised with offset mapping, the character positionof the final permitted token is read off, and the original text is slicedthere. The stored field is therefore always a verbatim prefix of the source.### Text cleaning`clean_text` removes URLs, HTML tags and redundant whitespace. Section 6.1and 6.2 quantify what these two substitutions remove.

In [7]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# Offset mapping requires a fast (Rust-backed) tokenizer.
assert tok.is_fast, "A fast tokenizer is required for offset-based truncation"

TOKEN_BUDGET = 245     # 256-token window less [CLS]/[SEP] and a safety margin
TITLE_BUDGET = 60
LABEL_BUDGET = 15
BODY_BUDGET  = 120     # design parameter - see markdown above


def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'http\S+', '', text)      # remove URLs
    text = re.sub(r'<.*?>', '', text)        # remove HTML tags
    text = re.sub(r'\s+', ' ', text).strip()  # collapse whitespace
    return text


def fit_to_tokens(text, limit):
    """Truncate `text` to at most `limit` tokens, returning (text, token_count).

    The result is always a verbatim prefix of the input. Truncation is done by
    reading the character offset of the last permitted token and slicing the
    original string - never by decoding token ids, which this tokenizer cannot
    do losslessly (it is uncased and splits on punctuation).
    """
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = enc['input_ids']
    if len(ids) <= limit:
        return text, len(ids)

    end_char = enc['offset_mapping'][limit - 1][1]
    out = text[:end_char]
    return out, len(tok.encode(out, add_special_tokens=False))


def build_issue_text(row):
    """Serialise one issue, keeping the result inside the encoder window."""
    parts, used = [], 0

    seg, n = fit_to_tokens(f"TITLE: {clean_text(row['title'])}", TITLE_BUDGET)
    parts.append(seg)
    used += n

    if pd.notna(row.get('labels_0_name')):
        seg, n = fit_to_tokens(f"LABEL: {row['labels_0_name']}", LABEL_BUDGET)
        parts.append(seg)
        used += n

    seg, n = fit_to_tokens(f"BODY: {clean_text(row['body'])}", BODY_BUDGET)
    parts.append(seg)
    used += n

    # Replies fill the remaining budget in thread order.
    # range(6) matches the six reply slots retained in the working dataset
    # and the six slots read by author authority scoring.
    for i in range(6):
        author = row.get(f'answers_{i}_author')
        body   = row.get(f'answers_{i}_body')
        if pd.isna(author) or pd.isna(body):
            continue
        headroom = TOKEN_BUDGET - used
        if headroom < 20:        # too little left to carry meaning
            break
        seg, n = fit_to_tokens(
            f"REPLY_{i+1} by {author}: {clean_text(body)}", headroom)
        parts.append(seg)
        used += n

    return '\n'.join(parts)

In [8]:
print("Building combined text fields... (2-3 minutes)")
df['combined_text'] = df.apply(build_issue_text, axis=1)
print("Done\n")

print("=== EXAMPLE COMBINED TEXT ===")
print(df['combined_text'].iloc[1])

Token indices sequence length is longer than the specified maximum sequence length for this model (674 > 512). Running this sequence through the model will result in indexing errors


Building combined text fields... (2-3 minutes)
Done

=== EXAMPLE COMBINED TEXT ===
TITLE: Able to change the window height even if IsResizable is false but ExtendsContentIntoTitleBar is true
LABEL: bug
BODY: ### Describe the bug If you set ExtendsContentIntoTitleBar to true you'll still be able to change a window's height when IsResizable is false. ### Steps to reproduce the bug Set presenter.IsResizable to false and ExtendsContentIntoTitleBar to true and try to change the window's height with the cursror. ### Expected behavior _No response_ ### Screenshots _No response_ ### NuGet package version WinUI 3 - Windows App SDK 1
REPLY_2 by codendone: This is very similar to #9670, though that issue is only talking about the cursor being incorrect. I'm not able to actually resize the window **when using 1.5**, but I do repro the issue with **1.6-experimental1**.
REPLY_3 by codendone: This was closed as fixed/duplicate of #7629. The fix will be


In [9]:
# Integrity checks - the field must be a faithful serialisation of its source.

# 1. Field markers survive construction with their casing intact.
slot_filled = pd.concat(
    [df[f'answers_{i}_author'].notna() & df[f'answers_{i}_body'].notna()
     for i in range(6)], axis=1).any(axis=1)
marker_present = df['combined_text'].str.contains('REPLY_', regex=False)

print(f"Issues with a populated reply slot: {slot_filled.sum():,}")
print(f"Issues whose text carries REPLY_:   {marker_present.sum():,}")
assert not (marker_present & ~slot_filled).any(), "spurious reply marker"
assert not (slot_filled & ~marker_present).any(), "reply marker lost in construction"

# 2. Title casing is preserved - a decode round-trip would lowercase it.
sample_idx = df.index[df['title'].str.contains(r'[A-Z]', na=False)][:200]
preserved = sum(
    f"TITLE: {clean_text(df.at[i, 'title'])}"[:40] in df.at[i, 'combined_text']
    for i in sample_idx)
print(f"Titles preserved verbatim:          {preserved} of {len(sample_idx)}")
assert preserved == len(sample_idx), "title text altered during construction"

print("\nIntegrity checks passed")

Issues with a populated reply slot: 15,722
Issues whose text carries REPLY_:   15,722
Titles preserved verbatim:          200 of 200

Integrity checks passed


In [10]:
df.to_csv('data/github_issues_cleaned.csv', index=False)
print("Cleaned dataset saved:", df.shape)
assert df.shape == (15955, 133), f"Unexpected shape: {df.shape}"

Cleaned dataset saved: (15955, 133)


## 4. Column reductionThe raw export carries 132 columns, most of them API metadata with no role inretrieval (label colours and URLs, reaction breakdowns, assignee identifiers).The working dataset retains 37: issue content, the first six reply slots,three label columns, and the metadata fields used for weighting or reporting.Reply slots 7-10 are dropped. They are populated in under 9% of issues, andthe token budget in Section 3 never reaches them.**Row order is preserved.** Downstream FAISS positions map directly ontodataframe indices, so any reordering or filtering here would break retrieval.

In [11]:
keep_cols = [
    # Issue identity
    'id', 'repo_name', 'title', 'body', 'state', 'state_reason',
    # Reporter
    'user_login', 'user_type',
    # Timestamps
    'created_at', 'closed_at', 'updated_at',
    # Labels (dynamic weighting + partitioning)
    'labels_0_name', 'labels_1_name', 'labels_2_name',
    # Milestone (sparse, retained for reference)
    'milestone_title',
    # Assignee
    'assignee_login',
    # Conversation thread - first six reply slots
    'answers_0_author', 'answers_0_body', 'answers_0_creation_time',
    'answers_1_author', 'answers_1_body', 'answers_1_creation_time',
    'answers_2_author', 'answers_2_body', 'answers_2_creation_time',
    'answers_3_author', 'answers_3_body', 'answers_3_creation_time',
    'answers_4_author', 'answers_4_body', 'answers_4_creation_time',
    'answers_5_author', 'answers_5_body', 'answers_5_creation_time',
    # Engagement signals
    'reactions_total_count', 'comments',
    # Serialised representation
    'combined_text',
]

keep_cols = [c for c in keep_cols if c in df.columns]
df_clean = df[keep_cols].copy()

print("Original columns:", df.shape[1])
print("Retained columns:", df_clean.shape[1])
print("Rows unchanged:  ", df_clean.shape[0] == df.shape[0])

Original columns: 133
Retained columns: 37
Rows unchanged:   True


In [12]:
print("=== FINAL DATASET SUMMARY ===")
print("Shape:", df_clean.shape)
print("\nMissing values by column:")
nulls = df_clean.isnull().sum()
print(nulls[nulls > 0])

=== FINAL DATASET SUMMARY ===
Shape: (15955, 37)

Missing values by column:
labels_0_name                  9
labels_1_name               7517
labels_2_name              12172
milestone_title            13546
assignee_login             12497
answers_0_author             682
answers_0_body               682
answers_0_creation_time      682
answers_1_author            3937
answers_1_body              3937
answers_1_creation_time     3937
answers_2_author            7334
answers_2_body              7334
answers_2_creation_time     7334
answers_3_author            9831
answers_3_body              9831
answers_3_creation_time     9831
answers_4_author           11678
answers_4_body             11678
answers_4_creation_time    11678
answers_5_author           12984
answers_5_body             12984
answers_5_creation_time    12984
dtype: int64


In [13]:
df_clean.to_csv('data/github_issues_final.csv', index=False)

print("Final dataset saved:", df_clean.shape)
assert df_clean.shape == (15955, 37), f"Unexpected shape: {df_clean.shape}"
assert df_clean.index.equals(pd.RangeIndex(15955)), "Row order altered"
print("Row order preserved - FAISS index alignment intact")

Final dataset saved: (15955, 37)
Row order preserved - FAISS index alignment intact


## 5. Post-construction validationThree checks confirm the constructed field is well formed and that the tokenbudget achieved what it was introduced for.

In [14]:
# 5.1 Character-level sanity checks
lengths = df_clean['combined_text'].str.len()

print("Issues with text under 50 characters:", (lengths < 50).sum())
print("Mean length:  ", int(lengths.mean()))
print("Median length:", int(lengths.median()))
print("Min / Max:    ", lengths.min(), "/", lengths.max())

Issues with text under 50 characters: 0
Mean length:   787
Median length: 821
Min / Max:     135 / 1835


In [15]:
# 5.2 Token lengths against the encoder window
sample = df_clean['combined_text'].head(500)
token_lens = [len(tok.encode(t, add_special_tokens=False)) for t in sample]

print("Encoder window:", 256, "tokens")
print("Sample of", len(sample), "issues -")
print("  mean:  ", int(np.mean(token_lens)))
print("  median:", int(np.median(token_lens)))
print("  max:   ", max(token_lens))
print("  over budget (245):", sum(l > TOKEN_BUDGET for l in token_lens))

Encoder window: 256 tokens
Sample of 500 issues -
  mean:   214
  median: 236
  max:    245
  over budget (245): 0


In [16]:
# 5.3 Thread coverage: does reply content survive into the embedded window?

def thread_survives(text, limit=256):
    """True if reply content falls inside the first `limit` tokens.

    Counts the tokens that PRECEDE the first reply marker. Decoding the
    truncated prefix and searching it for 'REPLY_' does not work: this
    tokenizer is uncased, so the marker comes back as 'reply _ 1'.
    """
    pos = text.find('REPLY_')
    if pos == -1:
        return False
    return len(tok.encode(text[:pos], add_special_tokens=False)) < limit


def build_issue_text_unbudgeted(row):
    """The unconstrained concatenation, retained for comparison only."""
    parts = [f"TITLE: {clean_text(row['title'])}",
             f"BODY: {clean_text(row['body'])}"]
    if pd.notna(row.get('labels_0_name')):
        parts.append(f"LABEL: {row['labels_0_name']}")
    for i in range(6):
        a, b = row.get(f'answers_{i}_author'), row.get(f'answers_{i}_body')
        if pd.notna(a) and pd.notna(b):
            parts.append(f"REPLY_{i+1} by {a}: {clean_text(b)}")
    return '\n'.join(parts)


rows = df_clean.head(500)
unbudgeted = rows.apply(build_issue_text_unbudgeted, axis=1)

has_reply = rows['combined_text'].str.contains('REPLY_')

print(f"Issues carrying reply content:          {has_reply.sum()} of {len(rows)}")
print(f"  reply survives - unbudgeted build:    {(has_reply & unbudgeted.apply(thread_survives)).sum()}")
print(f"  reply survives - budgeted build:      {(has_reply & rows['combined_text'].apply(thread_survives)).sum()}")

Issues carrying reply content:          495 of 500
  reply survives - unbudgeted build:    374
  reply survives - budgeted build:      495


## 6. Data quality auditsRead-only diagnostics supporting the methodology. These cells quantify whatthe preprocessing decisions above cost, and modify nothing.

In [17]:
# 6.1 HTML stripping: how much is genuine markup vs. technical notation?
bodies = df_clean['body'].dropna().astype(str)
hits = bodies.str.contains(r'<[A-Za-z_][\w:]*>')

print(f"Bodies containing angle-bracket tokens: {hits.sum()} of {len(bodies)} "
      f"({hits.mean()*100:.1f}%)")
print("\nMost frequent tokens removed:")
print(bodies[hits].str.findall(r'<[A-Za-z_][\w:]*>').explode().value_counts().head(15))

Bodies containing angle-bracket tokens: 933 of 15955 (5.8%)

Most frequent tokens removed:
body
<anonymous>    70
<summary>      60
<T>            58
<details>      52
<module>       49
<kbd>          45
<p>            38
<SNR>          38
<script>       37
<div>          36
<sup>          33
<string>       29
<int>          22
<leader>       17
<br>           17
Name: count, dtype: int64


In [18]:
# 6.2 URL removal: what kind of references are discarded?
has_url = bodies.str.contains(r'http\S+')
print(f"Bodies containing URLs: {has_url.sum()} of {len(bodies)} "
      f"({has_url.mean()*100:.1f}%)\n")

urls = bodies.str.findall(r'https?://[^\s)\]]+').explode().dropna()

def url_kind(u):
    u = u.lower()
    if 'user-images.githubusercontent' in u or u.endswith(('.png', '.jpg', '.jpeg', '.gif')):
        return 'image/screenshot'
    if re.search(r'github\.com/.+/(issues|pull)/\d+', u):
        return 'github issue/PR'
    if re.search(r'github\.com/.+/commit/', u):
        return 'github commit'
    if 'github.com' in u:
        return 'github other'
    return 'external'

print(urls.apply(url_kind).value_counts())

xrefs = bodies.str.contains(r'#\d+').sum()
print(f"\nBodies retaining #NNNN cross-references: {xrefs}")

Bodies containing URLs: 9984 of 15955 (62.6%)

body
github other        7880
external            6697
image/screenshot    3244
github issue/PR     1243
github commit        246
Name: count, dtype: int64

Bodies retaining #NNNN cross-references: 995


In [19]:
# 6.3 Bot filter false positives
# Human accounts whose login happens to contain a matched substring.
likely_human = {
    'dabbott', 'psobot', 'henrebotha', 'claudiobottari', 'bricebottegal',
    'sbottelli', 'talbotja', 'paleolimbot', 'nodebotanist', 'iBotPeaches',
    'SoggyBottomBoy', 'xobotyi', 'tibotiber', 'ibotty', 'botovq', 'rabotond',
    'MrRobotjs', 'greenrobot-team', 'DragonflyRobotics', 'IRobot1',
    'IndusRobot', 'MINIONBOTS', 'TheBotlyNoob', 'robotscissors', 'pixelbotme',
    'botne', 'botstar-tra', 'qunbotop', 'tronboto', 'JaimeFLandbot',
}

raw_audit = pd.read_csv('data/github_issues_tickets.csv', low_memory=False)
author_cols = [f'answers_{i}_author' for i in range(10)]

wrongly_removed = sum(raw_audit[c].isin(likely_human).sum() for c in author_cols)
affected_issues = raw_audit[author_cols].isin(likely_human).any(axis=1).sum()

authority_authors = set(pd.read_csv('data/author_authority_scores.csv')['author'])

print(f"Human accounts identified:  {len(likely_human)} of "
      f"{len(bot_matched_accounts)} matched accounts")
print(f"Replies wrongly removed:    {wrongly_removed} of {total_bots} removals "
      f"({wrongly_removed/total_bots*100:.1f}%)")
print(f"Issues affected:            {affected_issues} of {len(raw_audit)} "
      f"({affected_issues/len(raw_audit)*100:.2f}% of the corpus)")
print(f"Still in authority table:   "
      f"{[n for n in likely_human if n in authority_authors]}")

Human accounts identified:  30 of 97 matched accounts
Replies wrongly removed:    67 of 2520 removals (2.7%)
Issues affected:            36 of 15955 (0.23% of the corpus)
Still in authority table:   []


In [20]:
# 6.4 Resolution status - issues closed without a fix
print(df_clean['state'].value_counts(dropna=False))
print()
print(df_clean['state_reason'].value_counts(dropna=False))

not_planned = (df_clean['state_reason'] == 'not_planned').sum()
print(f"\nClosed without resolution: {not_planned} "
      f"({not_planned/len(df_clean)*100:.1f}%) - retained in the corpus")

state
closed    15955
Name: count, dtype: int64

state_reason
completed      14997
not_planned      958
Name: count, dtype: int64

Closed without resolution: 958 (6.0%) - retained in the corpus


In [21]:
# 6.5 Thread depth after bot filtering
print("Issues populated at each reply position:")
for i in range(6):
    n = df_clean[f'answers_{i}_body'].notna().sum()
    print(f"  Reply {i+1}: {n:>6,}  ({n/len(df_clean)*100:5.1f}%)")

Issues populated at each reply position:
  Reply 1: 15,273  ( 95.7%)
  Reply 2: 12,018  ( 75.3%)
  Reply 3:  8,621  ( 54.0%)
  Reply 4:  6,124  ( 38.4%)
  Reply 5:  4,277  ( 26.8%)
  Reply 6:  2,971  ( 18.6%)


In [22]:
import pandas as pd
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
tok   = model.tokenizer
LIMIT = model.max_seq_length          # 256

def encoder_view(text):
    """The prefix of `text` the encoder actually sees after its own truncation,
    plus the token count INCLUDING [CLS] and [SEP]."""
    t = str(text)
    enc = tok(t, truncation=True, max_length=LIMIT,
              return_offsets_mapping=True, add_special_tokens=True)
    spans = [o for o in enc['offset_mapping'] if o[1] > o[0]]   # drop specials
    return (t[:spans[-1][1]] if spans else ''), len(enc['input_ids'])

old = pd.read_csv('_pre_budget/data/github_issues_final.csv', low_memory=False)
new = pd.read_csv('data/github_issues_final.csv', low_memory=False)

N = 500
for name, d in (('UNBUDGETED (old)', old), ('BUDGETED (new)', new)):
    views = [encoder_view(t) for t in d['combined_text'].head(N)]
    toks  = [n for _, n in views]
    has   = d['combined_text'].head(N).str.contains('REPLY_', regex=False)
    print(name)
    print(f"  tokens incl. specials - max {max(toks)}, over {LIMIT}: "
          f"{sum(t > LIMIT for t in toks)}")
    print(f"  issues with reply content:        {has.sum()} of {N}")
    print(f"  reply content VISIBLE to encoder: {sum('REPLY_' in v for v, _ in views)} of {N}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


UNBUDGETED (old)
  tokens incl. specials - max 256, over 256: 0
  issues with reply content:        495 of 500
  reply content VISIBLE to encoder: 372 of 500

BUDGETED (new)
  tokens incl. specials - max 247, over 256: 0
  issues with reply content:        495 of 500
  reply content VISIBLE to encoder: 495 of 500



In [23]:
improved = [i for i in range(N)
            if 'REPLY_' in str(old.at[i, 'combined_text'])
            and 'REPLY_' not in encoder_view(old.at[i, 'combined_text'])[0]
            and 'REPLY_' in encoder_view(new.at[i, 'combined_text'])[0]]
print(f"{len(improved)} of the first {N} issues gained thread visibility\n")

for i in improved[:2]:
    print('=' * 78)
    print(f"ROW {i}")
    for label, d in (('OLD - what the encoder saw', old),
                     ('NEW - what the encoder sees', new)):
        v, n = encoder_view(d.at[i, 'combined_text'])
        print(f"\n--- {label}  ({n} tokens) ---")
        print(v)
    print()

# A short issue, where no truncation applies - confirms nothing is damaged
s = new['combined_text'].str.len().idxmin()
v, n = encoder_view(new.at[s, 'combined_text'])
print('=' * 78)
print(f"ROW {s} - shortest issue, {n} tokens, untruncated")
print('=' * 78)
print(v)
print(f"\nidentical to the stored field: {v == new.at[s, 'combined_text']}")

123 of the first 500 issues gained thread visibility

ROW 4

--- OLD - what the encoder saw  (256 tokens) ---
TITLE: MediaSession play/pause events not propagated to LeanbackPlayerAdapter
BODY: ### ExoPlayer Version 2.18.0 ### Devices that reproduce the issue Any Android TV ### Devices that do not reproduce the issue _No response_ ### Reproducible in the demo app? Yes ### Reproduction steps Connect player with a MediaSession and try to play/pause the video using mobile. ### Expected result When clicking play/pause the controls overlay should be triggered and their state updated. It seems that the `LeanbackPlayerAdapter` [listens]( to `onPlaybackStateChanged` player events, but play/pause action called from `MediaSessionConnector.` directly on a `Player` instance triggers only `onPlayWhenReadyChanged` event, which is not handled in the `LeanbackPlayerAdapter`. By using `ForwadingPlayer` when setting player for the `MediaSessionConnector.` you can workaround this issue: ``` mediaSessionC

In [24]:
import pandas as pd
df = pd.read_csv('data/github_issues_final.csv', low_memory=False)

reply_count = df[[f'answers_{i}_body' for i in range(6)]].notna().sum(axis=1)

cand = df[
    df['labels_0_name'].notna()
    & reply_count.between(2, 3)
    & df['combined_text'].str.count('REPLY_').ge(2)
    & df['combined_text'].str.len().between(600, 900)
]

print(f"{len(cand)} candidates\n")
for idx in cand.index[:5]:
    n = len(tok.encode(df.at[idx, 'combined_text'], add_special_tokens=True))
    print("=" * 78)
    print(f"ROW {idx}  |  {len(df.at[idx, 'combined_text'])} chars  |  {n} tokens"
          f"{'  (budget reached - truncated)' if n >= 247 else '  (complete)'}")
    print("=" * 78)
    print(df.at[idx, 'combined_text'])
    print()

3067 candidates

ROW 0  |  819 chars  |  229 tokens  (complete)
TITLE: Article mistake
LABEL: help wanted
BODY: ### Describe the problem that you experienced I think there is a samll mistake in the tutorial in: Step 3 - Add features to HomeComponent Point 2. In `app.component.ts`, in @[Component]( update the template property with this code. I guess it should be `home.component.ts` It is a minor mistake, but can be misleading for newbes like me ### Enter the URL of the topic with the problem ### Describe what you were looking for in the documentation _No response_ ###
REPLY_1 by MukulKolpe: Hey @alan-agius4, can I work on this issue?
REPLY_2 by JeanMeche: @MukulKolpe You can open a PR if you'd like to fix that issue 😊
REPLY_3 by jamesanderson9182: Following the history and looking at the page, it looks like this issue can be closed 😄 ![Screenshot 2023-05-16 at 22 51 28](

ROW 1  |  868 chars  |  247 tokens  (budget reached - truncated)
TITLE: Able to change the window height even if Is

In [25]:
rec = df.at[14, 'combined_text']
print(rec)
print()
print(f"[{len(rec)} characters, "
      f"{len(tok.encode(rec, add_special_tokens=True))} tokens]")

TITLE: Build fails: Can't find amdhip64 and roctracer libs
LABEL: bug
BODY: ### Description Hello, When building v0.4.16 on Arch Linux, I ran into an issue where the amdhip64 and roctracer libs can't be found. It seems like JAX is looking for them under `/opt/rocm/hip` and `/opt/rocm/roctracer`, respectively. However, both libs only exist in `/opt/rocm/lib`. Copying the libs to where JAX is looking for them resolves the issue, but I think that work should be handled by
REPLY_1 by hawkinsp: v0.4.16 is quite old at this point. Does the issue still happen with a current version?
REPLY_2 by brianrobt: @hawkinsp, Let me try early next week and I'll get back to you. Thanks!
REPLY_3 by brianrobt: The issue doesn't happen on v0.4.29. I'm closing out the issue. Thanks @hawkinsp!

[780 characters, 238 tokens]


In [26]:
raw = pd.read_csv('data/github_issues_tickets.csv', low_memory=False)
for i in range(6):
    m = raw[f'answers_{i}_author'] == 'elasticmachine'
    for b in raw.loc[m, f'answers_{i}_body'].head(3):
        print(str(b)[:200], "\n---")

Pinging @elastic/ml-ui (:ml) 
---
Pinging @elastic/apm-ui (Team:apm) 
---
Pinging @elastic/appex-sharedux (Team:SharedUX) 
---
Pinging @elastic/kibana-app-services (Team:AppServicesUx) 
---
Pinging @elastic/actionable-observability (Team: Actionable Observability) 
---
Pinging @elastic/kibana-app-arch (Team:AppArch) 
---
Pinging @elastic/security-threat-hunting (Team:Threat Hunting) 
---
Pinging @elastic/kibana-app-services (Team:AppServices) 
---
Pinging @elastic/es-core-infra 
---
Pinging @elastic/kibana-data-discovery (Team:DataDiscovery) 
---
Pinging @elastic/kibana-app 
---
Pinging @elastic/kibana-app-services (Team:AppServicesSv) 
---
Pinging @elastic/kibana-app-services (Team:AppServices) 
---
Pinging @elastic/kibana-core (Team:Core) 
---
